# Find users by name

## What it does

Shows the users saved with that name, lowest number first. Spaces around the name do not matter; capitals do.

## What happens to the saved users

It creates two practice users with the same name, reads them, and deletes them at the end. The real users stay the same.

## Feature it uses

`findUserByName(name)`, from `logic/find-user-by-name.js`.

## Failures it can show

- A name nobody has, or the same name in other capitals → only the header.
- An empty name → `Name is empty`.
- The users cannot be read → `Could not find the user`.

Open it with `pnpm jupyter` from `first-app`.

In [1]:
import os
import subprocess
from pathlib import Path

app = Path.cwd()
while app != app.parent and not (app / "logic").is_dir():
    app = app.parent

code = r"""
try {
  const { default: Table } = await import("cli-table3");
  const { createUser } = await import("./logic/create-user.js");
  const { findUserByName } = await import("./logic/find-user-by-name.js");
  const { deleteUser } = await import("./logic/delete-user.js");

  function printUsers(users) {
    const table = new Table({
      head: ["id", "name", "email", "createdAt", "updatedAt"],
      style: { head: ["cyan", "bold"] },
    });
    for (const user of users) {
      table.push([user.id, user.name, user.email, user.createdAt, user.updatedAt]);
    }
    console.log(table.toString());
  }

  const name = `Practice ${Date.now()}`;
  const first = createUser(name, `practice-${Date.now()}-1@x.com`);
  try {
    const second = createUser(name, `practice-${Date.now()}-2@x.com`);
    try {
      // 1. Find the name, with spaces around: a table with both practice users, lowest id first
      printUsers(findUserByName(`  ${name}  `));

      // 2. The same name in lowercase: only the header
      printUsers(findUserByName(name.toLowerCase()));

      // 3. Only spaces: "Name is empty"
      try {
        findUserByName("   ");
      } catch (error) {
        console.log(error.message);
      }
    } finally {
      deleteUser(second.id);
    }
  } finally {
    deleteUser(first.id);
  }
} catch (error) {
  console.log(error.message);
}
"""

if (app / "logic").is_dir():
    env = os.environ.copy()
    env["FORCE_COLOR"] = "1"
    env.pop("NO_COLOR", None)
    completed = subprocess.run(
        ["node", "--no-warnings", "--input-type=module", "-e", code],
        cwd=app,
        capture_output=True,
        text=True,
        env=env,
    )
    print(completed.stdout, end="")
    if completed.returncode != 0:
        print(completed.stderr, end="")
else:
    print("Cannot find first-app.")

┌────┬────────────────────────┬────────────────────────────────┬─────────────────────┬─────────────────────┐
│ id │ name                   │ email                          │ createdAt           │ updatedAt           │
├────┼────────────────────────┼────────────────────────────────┼─────────────────────┼─────────────────────┤
│ 1  │ Practice 1793702627736 │ practice-1793702627736-1@x.com │ 2026-11-03 11:43:47 │ 2026-11-03 11:43:47 │
├────┼────────────────────────┼────────────────────────────────┼─────────────────────┼─────────────────────┤
│ 2  │ Practice 1793702627736 │ practice-1793702627738-2@x.com │ 2026-11-03 11:43:47 │ 2026-11-03 11:43:47 │
└────┴────────────────────────┴────────────────────────────────┴─────────────────────┴─────────────────────┘
┌────┬──────┬───────┬───────────┬───────────┐
│ id │ name │ email │ createdAt │ updatedAt │
└────┴──────┴───────┴───────────┴───────────┘
Name is empty
